# 04 — `multiprocessing` on WebAssembly: the one-line port

`wasm_multiprocessing` puts the `multiprocessing.Pool` API on top of the
same nested-worker pool the other notebooks use: change one import line
and code written for CPython processes runs on Pyodide Web Workers. This
notebook:

1. installs both wheels shipped with this site,
2. ports the classic count-primes `Pool.map` script (one import change +
   an awaited entry point),
3. tours `AsyncResult` (`apply_async`, `ready`, `successful`, awaited
   `aget`), including how worker exceptions re-raise,
4. shows the sync-method capability detection (JSPI stack switching),
5. times `Pool(1)` / `Pool(2)` / `Pool(4)` on one workload, with a bar
   chart.

> **Environment requirements** — same as the other notebooks: a
> cross-origin-isolated page (COOP/COEP headers, provided by
> `npm run serve:lite`), nested-worker support, and network access to the
> Pyodide CDN. Design + full API mapping:
> `docs/architecture/multiprocessing-shim-design.md`.

## Install both wheels

`wasm_multiprocessing` depends on `pyodide_pool` (which is not on PyPI),
so the pool wheel installs first. The paths are absolute for the same
reason as in `01-pool-basics.ipynb`: the kernel worker resolves relative
URLs against its own script URL, and this site is served at the domain
root.

In [ ]:
import piplite

await piplite.install("/files/wheels/pyodide_pool-0.1.0-py3-none-any.whl")
await piplite.install("/files/wheels/wasm_multiprocessing-0.1.0-py3-none-any.whl")

## Boot the worker pool

The shim does not spawn workers itself: `Pool(n)` rides the package-wide
default `pyodide_pool` pool, with `n` as a driver-side **soft cap** on
in-flight chunks. The JS pool's 4 workers are the hard concurrency cap
— that split is what makes timing `Pool(1)`/`Pool(2)`/`Pool(4)` on one
shared pool meaningful below.

In [ ]:
import pyodide_pool

js_pool = await pyodide_pool.create_pool(pool_size=4)
print("JS pool ready with", js_pool.pool_size, "workers")

## The classic script, ported

Written against the stdlib, the count-primes example looks like this:

```python
from multiprocessing import Pool

def count_primes(bounds):
    ...

def main():
    with Pool(4) as pool:
        counts = pool.map(count_primes, RANGES)
    return sum(counts)

print(main())
```

Two mechanical edits make it run here: swap the import line, and make the
entry point async — awaiting the `amap` form (under JSPI the original
`pool.map` line runs verbatim instead; see the capability-detection
section below). `with Pool(...)` keeps its stdlib meaning: on exit the
pool is terminated, which here kills the JS pool's idle workers — so a
later section re-warms them before timing anything.

In [ ]:
from wasm_multiprocessing import Pool  # was: from multiprocessing import Pool

RANGES = [(2, 25_000), (25_000, 50_000), (50_000, 75_000), (75_000, 100_000)]
EXPECTED_PORT_TOTAL = 9_592  # π(100_000)


def count_primes(bounds):
    lo, hi = bounds
    count = 0
    for n in range(lo, hi):
        if n < 2:
            continue
        if n == 2:
            count += 1
            continue
        if n % 2 == 0:
            continue
        d = 3
        is_prime = True
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 2
        if is_prime:
            count += 1
    return count


async def main():  # was: def main()
    with Pool(4) as pool:
        counts = await pool.amap(count_primes, RANGES)  # was: pool.map(...)
    return sum(counts)


port_total = await main()  # was: main()
assert port_total == EXPECTED_PORT_TOTAL, port_total
print(f"ported script counted {port_total:,} primes below 100,000")

## `AsyncResult`

The `*_async` methods return a stdlib-shaped `AsyncResult` backed by an
asyncio task. `ready()`/`successful()` are non-blocking; the portable
completion form is the awaitable `aget()` (`await result` is sugar for
it), while blocking `get()`/`wait()` need JSPI — next section. The
`with` block above terminated the pool's workers, so the first task here
pays one fresh worker boot.

In [ ]:
mp = Pool(4)

result = mp.apply_async(count_primes, ((2, 50_000),))
print("ready() right after submit:", result.ready())

value = await result.aget()
print("value:", value, "| ready():", result.ready(), "| successful():", result.successful())

### Worker exceptions re-raise faithfully

A failing task re-raises its **original** exception type in the kernel,
with the worker-side traceback chained as `RemoteTraceback` — so
`except` clauses written for stdlib `multiprocessing` keep working.

In [ ]:
failing = mp.apply_async(count_primes, ("not a range",))

try:
    await failing.aget()
except Exception as err:
    print("re-raised:", type(err).__name__, "-", err)
    print("chained cause:", type(err.__cause__).__name__)
print("successful():", failing.successful())

## Sync methods: capability detection

The blocking stdlib methods (`map`, `starmap`, `apply`, `join`,
`AsyncResult.get`/`wait`) work only where the runtime can suspend
WebAssembly frames — JSPI stack switching, detected **per call** via
`pyodide.ffi.can_run_sync()`. Where it is unavailable they raise a
`RuntimeError` naming the exact async replacement instead of
deadlocking. This cell prints whichever behavior your browser has
(Chrome 137+ ships JSPI on by default).

In [ ]:
from pyodide.ffi import can_run_sync

print("can_run_sync():", can_run_sync())
try:
    sync_counts = mp.map(count_primes, [(2, 1_000)])
    print("pool.map blocked and returned:", sync_counts)
except RuntimeError as err:
    print("pool.map raised RuntimeError:")
    print(err)

## Warm all workers

The timing below should measure compute + dispatch, not interpreter
boot: the `with` block earlier terminated the original workers and the
cells since re-booted at most one, so force all four to boot now.

In [ ]:
import asyncio

await asyncio.gather(*(js_pool.submit(lambda i=i: i) for i in range(js_pool.pool_size)))
print("all", js_pool.pool_size, "workers warm")

## Time `Pool(1)` / `Pool(2)` / `Pool(4)`

The same workload for every size — π(1,000,000) in 8 chunks,
`chunksize=1` so each chunk is one task message. `Pool(n)` only caps
in-flight chunks, so `Pool(1)` is the serial baseline over the very same
dispatch path. (When the cap cannot bind tighter than the 4 JS workers
— `Pool(4)` here — the whole map goes out as one batched
`mapPickled` run instead of gated single submits.)

In [ ]:
import time

RANGE_END = 1_000_000
CHUNK_COUNT = 8
EXPECTED_TOTAL = 78_498  # π(1_000_000)

width = -(-(RANGE_END - 2) // CHUNK_COUNT)  # ceiling division
chunks = [(lo, min(lo + width, RANGE_END)) for lo in range(2, RANGE_END, width)]

timings = {}
for n in (1, 2, 4):
    p = Pool(n)
    t0 = time.perf_counter()
    counts = await p.amap(count_primes, chunks, chunksize=1)
    timings[n] = time.perf_counter() - t0
    total = sum(counts)
    assert total == EXPECTED_TOTAL, (n, total)
    print(f"Pool({n}): {timings[n]:.2f} s — {total:,} primes")

## Timings chart

(The first `matplotlib` import downloads the package into the kernel —
give it a few seconds.)

In [ ]:
import matplotlib.pyplot as plt

sizes = sorted(timings)
speedup = timings[1] / timings[4]

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(
    [f"Pool({n})" for n in sizes],
    [timings[n] for n in sizes],
    color=["#b3b2ab", "#2a78d6", "#008300"],
    width=0.55,
)
ax.bar_label(bars, fmt="%.2f s", padding=3)
ax.set_ylabel("wall-clock (s)")
ax.set_title(f"π(1,000,000) across {CHUNK_COUNT} chunks — Pool(4) is {speedup:.2f}× Pool(1)")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.grid(axis="y", color="#e3e2dc", linewidth=0.8)
ax.set_axisbelow(True)
plt.show()

## Success marker

The e2e smoke test asserts this exact line; it doubles as the notebook's
own bottom-line check.

In [ ]:
assert port_total == EXPECTED_PORT_TOTAL, port_total
assert timings[4] < timings[1], timings
print("MP_DEMO_OK", round(timings[1] / timings[4], 2))